# PhaseNet-DAS picks -> PyOcto association (run that produced `events_octo_das.csv`)

Reconstructed 2026-11-02 from the IPython kernel history (`~/.ipython/profile_default/history.sqlite`,
session 1703, executed 2026-10-29 16:54; the full run finished 2026-10-30 02:01). The cells below are the
exact cells that were executed, in order. The later `*_bothwaves*` runs edited
`picking_rainier_seismicstations_fromQibin_10292026.ipynb` in place, so this file keeps the original settings.

Outputs (in `.../phasenet-3-month-run_08182026/results/events_phasenet_das/`):
`events_octo_das.csv` (782 events), `assignments_octo_das.csv`, `all_pick_assignments_octo_das.csv`.
These are the events rendered in `pyocto_pick_plots/plots_das_all/` (and the 71-event subset in `plots_das/`).

Settings:

| item | value |
| --- | --- |
| picks | all `decimator_2023*.csv` PhaseNet-DAS minute files, every 10th channel (320 located channels used) |
| stations | `data/Mora_channellocation.csv`, channels with a surveyed position |
| velocity model | 1-D PyOcto table `data/vmodel/velocity_model` (IASP91, first 78 layers of `data/vmodel/IASP91_1.csv`, prebuilt Sep 21 with `create_model(layers, 1., 600, 400)`), `tolerance=2.0` |
| area | lat 45-48, lon -123 to -120, depth 0-40 km, `time_before=300` |
| thresholds | `n_picks=100`, `n_p_picks=0`, `n_s_picks=0`, `n_p_and_s_picks=0` (no P/S requirement) |
| other | pyocto 0.2.0 defaults (min_node_size 10, pick_match_tolerance 1.5, min_interevent_time 3.0, refinement_iterations 3) |

Environment: `test_environment` conda env. Re-running the save cell overwrites the three CSVs above.


In [1]:
import os
import glob
import pandas as pd
from obspy import UTCDateTime

filepath = '/1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/phasenet-3-month-run_08182026/results/picks_phasenet_das/'

In [2]:
### Merge PhaseNet-DAS picks from all minute files
csv_list = glob.glob(filepath+'/decimator_2023*.csv')   # one hour, holds the M2.9 PNSN event at 10:10:23 UTC
all_csv =[]
for i in csv_list:
    if os.path.getsize(i) == 0:   # minutes with no picks are 0-byte files; read_csv raises on them
        continue
    all_csv.append(pd.read_csv(i))
df = pd.concat(all_csv, axis=0)
df = df[df['channel_index'] % 10 == 0]   # every 10th channel (~96 m spacing); neighbouring channels carry the same arrival

# change to Octo format
picks = df[['channel_index','phase_type','phase_time']].rename(
    columns={"channel_index": "station", "phase_type": "phase", "phase_time": "time"})
picks['time'] = (pd.to_datetime(picks['time'], utc=True, format='ISO8601') - pd.Timestamp(0, tz='UTC')).dt.total_seconds()   # same as UTCDateTime(x).timestamp, vectorised for 13.8 M rows
picks['station'] = picks['station'].astype(str)   # PyOcto matches picks to the station table on this id

picks.to_csv(filepath + 'picks_octo_das_11022026.csv', index=False)

In [3]:
picks

,station,phase,time
0,0,S,1.692922e+09
10,10,S,1.692922e+09
20,20,S,1.692922e+09
30,30,S,1.692922e+09
4,850,S,1.693094e+09
...,...,...,...
269,510,S,1.699660e+09
279,520,S,1.699660e+09
289,530,S,1.699660e+09
299,540,S,1.699660e+09


In [4]:
### Station table for PyOcto -- every DAS channel with a surveyed position is a station.
# Channels the survey could not locate have NaN coordinates and are dropped; their picks are
# dropped too, since PyOcto silently ignores picks whose station id is not in the table.
geom = pd.read_csv('data/Mora_channellocation.csv')
stations_table = pd.DataFrame({
    'id': geom['Channel'].astype(str),
    'longitude': geom['Longitude [°]'],
    'latitude': geom['Latitude [°]'],
    'elevation': geom['Elevation [m]'],
}).dropna(subset=['longitude', 'latitude']).reset_index(drop=True)

picks = picks[picks['station'].isin(stations_table['id'])].reset_index(drop=True)
print('%d stations, %d picks on located channels' % (len(stations_table), len(picks)))

3191 stations, 758167 picks on located channels


In [5]:
layers = pd.read_csv("data/vmodel/IASP91_1.csv").iloc[:78]
print(layers.dtypes)
print(layers.isna().sum())
print(layers.head())
print(layers.tail())
import os

import pyocto

velocity_model = pyocto.VelocityModel1D("data/vmodel/velocity_model", tolerance=2.0)

associator = pyocto.OctoAssociator.from_area(
    lat=(45, 48),
    lon=(-123, -120),
    zlim=(0, 40),
    time_before=300,
    velocity_model=velocity_model,
    n_picks=100,
    n_p_picks=0,
    n_s_picks=0,
    n_p_and_s_picks=0,
)

# Adds the x/y/z columns (km, in the associator's local frame) that associate() requires.
# This used to be safe to skip only because stations_table was loaded from the cached
# stations_table.csv, which was already transformed; the table is rebuilt from the two
# inventories on every run now, so it arrives with lon/lat/elev only.
associator.transform_stations(stations_table)
# stations_table.to_csv(filepath + 'stations_table.csv', index=False)
stations_table.head(), picks.head()

depth    float64
r        float64
vp       float64
vs       float64
dtype: object
depth    0
r        0
vp       0
vs       0
dtype: int64
   depth       r   vp    vs
0    0.0  6371.0  5.8  3.36
1    1.0  6370.0  5.8  3.36
2    2.0  6369.0  5.8  3.36
3    3.0  6368.0  5.8  3.36
4    4.0  6367.0  5.8  3.36
    depth       r      vp      vs
73  360.0  6011.0  8.8475  4.7830
74  370.0  6001.0  8.8840  4.8004
75  380.0  5991.0  8.9205  4.8178
76  390.0  5981.0  8.9570  4.8352
77  400.0  5971.0  8.9935  4.8526


(  id  longitude  latitude  elevation          x          y      z
 0  0 -121.73694  46.78586     1667.0 -18.092578  31.804569 -1.667
 1  1 -121.73694  46.78583     1667.0 -18.092588  31.801234 -1.667
 2  2 -121.73695  46.78581     1667.0 -18.093358  31.799013 -1.667
 3  3 -121.73696  46.78579     1667.0 -18.094129  31.796792 -1.667
 4  4 -121.73697  46.78577     1667.0 -18.094899  31.794571 -1.667,
   station phase          time
 0       0     S  1.692922e+09
 1      10     S  1.692922e+09
 2      20     S  1.692922e+09
 3      30     S  1.692922e+09
 4     850     S  1.693094e+09)

In [6]:
# Association
events, assignments = associator.associate(picks, stations_table)

print('%d events from %d picks' % (len(events), len(picks)))
if len(events):   # an empty result has no columns at all, so transform/convert would KeyError
    associator.transform_events(events)
    events['time'] = events['time'].apply(UTCDateTime)
all_pick_assignments = pd.merge(events, assignments, left_on="idx", right_on="event_idx", suffixes=("", "_pick"))

778 events from 758167 picks


In [7]:
### Save association results
# events_phasenet_das/ sits next to picks_phasenet_das/ in the PhaseNet-DAS results folder.
out_path = filepath.replace('picks_phasenet_das', 'events_phasenet_das')
os.makedirs(out_path, exist_ok=True)

# UTCDateTime objects would be written via str(); isoformat gives a round-trippable timestamp
events_out = events.copy()
if len(events_out):
    events_out['time'] = events_out['time'].apply(lambda t: t.isoformat())
events_out.to_csv(out_path + 'events_octo_das_11022026.csv', index=False)

assignments.to_csv(out_path + 'assignments_octo_das_11022026.csv', index=False)   # raw pick -> event_idx table

all_out = all_pick_assignments.copy()
if len(all_out):
    all_out['time'] = all_out['time'].apply(lambda t: t.isoformat())
    all_out['time_pick'] = pd.to_datetime(all_out['time_pick'], unit='s', utc=True)   # pick epoch seconds -> ISO timestamp
all_out.to_csv(out_path + 'all_pick_assignments_octo_das_11022026.csv', index=False)

print('saved %d events, %d assignments to %s' % (len(events_out), len(assignments), out_path))

saved 778 events, 245303 assignments to /1-fnp/cascadia/c-whd02/arose_data/arose-scratch-rainier/picking-denoising/phasenet-3-month-run_08182026/results/events_phasenet_das/


In [8]:
events

,idx,time,x,y,z,picks,latitude,longitude,depth
0,0,2023-08-23T19:18:55.116946Z,24.485643,-149.076192,39.375,105,45.158335,-121.188594,39.375
1,1,2023-08-23T19:36:27.322359Z,40.193413,-66.740058,39.375,375,45.898404,-120.982076,39.375
2,2,2023-08-23T19:36:30.742665Z,89.164699,-94.185436,0.625,177,45.646923,-120.356194,0.625
3,3,2023-08-24T06:56:23.275492Z,-18.941724,39.120686,0.625,640,46.851648,-121.748363,0.625
4,4,2023-08-24T09:04:10.261883Z,-25.409629,35.199918,1.875,512,46.816164,-121.832951,1.875
...,...,...,...,...,...,...,...,...,...
773,773,2023-11-20T22:55:20.567940Z,-24.485643,32.586072,0.625,458,46.792686,-121.820704,0.625
774,774,2023-11-21T01:56:12.968473Z,-24.485643,41.734532,3.125,508,46.874979,-121.821194,3.125
775,775,2023-11-21T18:47:09.543286Z,-68.836995,12.982231,0.625,108,46.613257,-122.398621,0.625
776,776,2023-11-21T19:16:26.004491Z,-18.017737,35.199918,0.625,299,46.816405,-121.736093,0.625
